# MAG7 SMA Crossover With Monte Carlo

This notebook runs the shared in-memory MAG7 SMA crossover job for each framework, then runs a separate Monte Carlo robustness job on the resulting equity curve.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from time import perf_counter

import pandas as pd

from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.sma_crossover import (
    run_sma_crossover_backtest as run_backtesting_py_runner,
)
from quant_orchestrator.platforms.backtesting_frameworks.nautilus.sma_crossover import (
    run_sma_crossover_backtest as run_nautilus_runner,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import MAG7_SYMBOLS, load_signal_frame, normalize_session_label
from quant_orchestrator.platforms.backtesting_frameworks.zipline.sma_crossover import (
    run_sma_crossover_backtest as run_zipline_runner,
)
from quant_orchestrator.strategy import summarize_backtest


SYMBOL_COUNT = len(MAG7_SYMBOLS)


def _wrap_runner(runner):
    def _run(frame: pd.DataFrame, *, symbol: str, capital_base: float):
        fast_window = int(frame.attrs.get('fast_window', 50))
        slow_window = int(frame.attrs.get('slow_window', 200))
        return runner(
            frame.loc[:, ['open', 'high', 'low', 'close', 'volume']].copy(),
            symbol=symbol,
            fast_window=fast_window,
            slow_window=slow_window,
            capital_base=capital_base,
        )

    return _run


run_backtesting_py = _wrap_runner(run_backtesting_py_runner)
run_zipline = _wrap_runner(run_zipline_runner)
run_nautilus = _wrap_runner(run_nautilus_runner)


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/backtesting/_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

...


Portfolio summary by framework:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second
0,backtesting.py,MAG7,2020-10-15,2026-06-23,1427,22,117026.96,0.1703,-0.0967,0.0036,0.1125,12684.44
1,zipline,MAG7,2020-10-15,2026-06-23,1427,44,117464.42,0.1746,-0.4422,0.0182,1.0932,1305.34
2,nautilus,MAG7,2020-10-15,2026-06-23,1427,40,116321.16,0.1632,-0.0974,0.0036,0.1670,8544.91



Summary by framework:


,framework,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,total_elapsed_seconds,total_trades
0,backtesting.py,0.1703,-0.0967,0.1125,0.1125,22
1,nautilus,0.1632,-0.0974,0.1670,0.1670,40
2,zipline,0.1746,-0.4422,1.0932,1.0932,44



=== AAPL ===
[backtesting.py]
Start                2020-10-15 00:00:00
End                  2026-06-23 00:00:00
Equity Final [$]            16212.474578
Return [%]                     13.487322
Sharpe Ratio                     0.28075
Max. Drawdown [%]             -13.582927
# Trades                               5
dtype: object

[zipline]
                           portfolio_value  returns
2026-06-16 20:00:00+00:00      7350.590566  0.00000
2026-06-17 20:00:00+00:00      7350.590566  0.00000
2026-06-18 20:00:00+00:00      7350.590566  0.00000
2026-06-22 20:00:00+00:00      7350.590566  0.00000
2026-06-23 20:00:00+00:00     15885.290212  1.16109

[nautilus]
                             side filled_qty  avg_px                   ts_last
client_order_id                                                               
O-20201015-000000-001-000-1   BUY         29  120.71 2020-10-15 00:00:00+00:00
O-20220603-000000-001-000-2  SELL         29  145.38 2022-06-03 00:00:00+00:00
O-20220928-000000

# Monte Carlo Robustness

The framework backtests above are the first job. This section is the second job: Monte Carlo simulation on the resulting portfolio returns.

No native Monte Carlo API was found in the documented surfaces for the three frameworks in this repo, so the notebook uses the orchestrator simulation for the comparison path.


In [2]:
from quant_orchestrator.monte_carlo import simulate_return_paths


MC_ITERATIONS = 1000
MC_BLOCK_SIZE = 5


def _portfolio_equity_for_framework(runs: dict[str, dict[str, object]], framework: str) -> pd.Series:
    sleeves = []
    for symbol in MAG7_SYMBOLS:
        sleeves.append(runs[symbol][framework][2].rename(symbol))
    combined_index = pd.Index([])
    for sleeve in sleeves:
        combined_index = combined_index.union(sleeve.index)
    combined = pd.Series(0.0, index=combined_index)
    for sleeve in sleeves:
        reindexed = sleeve.reindex(combined_index)
        reindexed = reindexed.ffill().fillna(sleeve.iloc[0])
        combined = combined.add(reindexed, fill_value=0.0)
    return combined.sort_index().rename("portfolio_value")


def run_monte_carlo_report(*, start: str = "2020-01-01", end: str | None = None):
    summary, portfolio_summary, runs = run_mag7_comparison(start=start, end=end)
    native_support = pd.DataFrame([
        {"framework": "backtesting.py", "native_monte_carlo": "no"},
        {"framework": "zipline", "native_monte_carlo": "no"},
        {"framework": "nautilus", "native_monte_carlo": "no"},
    ])

    mc_rows = []
    mc_results: dict[str, dict[str, object]] = {}
    for framework in ("backtesting.py", "zipline", "nautilus"):
        equity = _portfolio_equity_for_framework(runs, framework)
        returns = equity.pct_change().dropna()
        mc = simulate_return_paths(returns, iterations=MC_ITERATIONS, block_size=MC_BLOCK_SIZE, horizon=len(returns))
        mc_rows.append(
            {
                "framework": framework,
                "observations": len(returns),
                "iterations": int(mc.summary.loc[0, "iterations"]),
                "horizon": int(mc.summary.loc[0, "horizon"]),
                "terminal_return_mean": float(mc.summary.loc[0, "terminal_return_mean"]),
                "terminal_return_p05": float(mc.summary.loc[0, "terminal_return_p05"]),
                "terminal_return_p50": float(mc.summary.loc[0, "terminal_return_p50"]),
                "terminal_return_p95": float(mc.summary.loc[0, "terminal_return_p95"]),
                "max_drawdown_mean": float(mc.summary.loc[0, "max_drawdown_mean"]),
                "max_drawdown_p05": float(mc.summary.loc[0, "max_drawdown_p05"]),
            }
        )
        mc_results[framework] = {"equity": equity, "returns": returns, "mc": mc}

    mc_report = pd.DataFrame(mc_rows)
    return summary, portfolio_summary, native_support, mc_report, mc_results


summary, portfolio_summary, native_support, mc_report, mc_results = run_monte_carlo_report(start="2020-01-01")

print("Native Monte Carlo support:")
display(native_support.round(4))
print()
print("Monte Carlo robustness report:")
display(mc_report.round(4).head(10))
print()
for framework in ("backtesting.py", "zipline", "nautilus"):
    print(f"[{framework}] Monte Carlo summary")
    display(mc_results[framework]['mc'].summary.round(4))
    print()


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

Native Monte Carlo support:


...

,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05
0,1000,1426,0.174,-0.0302,0.1626,0.4093,-0.096,-0.1587



[zipline] Monte Carlo summary


,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05
0,1000,1426,0.4571,-0.5885,0.152,2.4238,-0.4194,-0.6963



[nautilus] Monte Carlo summary


,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05
0,1000,1426,0.167,-0.0409,0.1564,0.4036,-0.0982,-0.1635
